# 04 · 自动微分与一个小型 Poisson VAE
连续 Gaussian latent 的 reparameterization：$z=\mu+\sigma\epsilon$。这不能直接用于离散 blinking 状态。该 VAE 只教 ELBO/梯度，不替代物理 SMLM 模型。


In [ ]:
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(2); torch.manual_seed(7)
torch.set_default_dtype(torch.float64)
theta=torch.tensor(3.,requires_grad=True); y=torch.tensor(2.)
lp=torch.distributions.Poisson(theta).log_prob(y)
score=torch.autograd.grad(lp,theta)[0]
assert torch.allclose(score,y/theta-1)
print('score:',score.item())


In [ ]:
from torch import nn
N,P=256,8
basis=torch.stack([torch.linspace(.1,1,P),torch.linspace(1,.1,P)])
with torch.no_grad():
    latent=torch.randn(N,2)
    counts=torch.poisson(torch.nn.functional.softplus(latent@basis)+.2)
encoder=nn.Sequential(nn.Linear(P,16),nn.Tanh(),nn.Linear(16,4))
decoder=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,P),nn.Softplus())
opt=torch.optim.Adam(list(encoder.parameters())+list(decoder.parameters()),lr=.01)
history=[]
for step in range(150):
    mu,raw=encoder(torch.log1p(counts)).chunk(2,dim=-1)
    logvar=raw.clamp(-8,6)
    z=mu+torch.exp(.5*logvar)*torch.randn_like(mu)
    rate=decoder(z)+1e-6
    reconstruction=torch.distributions.Poisson(rate).log_prob(counts).sum(-1)
    kl=.5*(mu.square()+logvar.exp()-1-logvar).sum(-1)
    loss=-(reconstruction-kl).mean()
    opt.zero_grad(); loss.backward(); opt.step(); history.append(loss.item())
assert all(torch.isfinite(torch.tensor(history)))
plt.plot(history); plt.xlabel('training step'); plt.ylabel('negative MC ELBO');


## 检查
encoder 输出 q 的参数，decoder 输出 likelihood 的 rate；二者都不是“真实隐状态”。训练中的 MC ELBO 可上下波动。

实践任务：固定 epsilon，对一个 encoder 权重做 central finite difference，和 autograd 比较。不要对两次随机采样的 loss 做有限差分。

物理模型方向：保留 pixel-integrated PSF + Poisson decoder、Markov prior；网络预测识别势或 variational 参数。可解释性仍需校准、错设检验、参数可辨识性与 posterior diagnostics，并不由 physics decoder 自动保证。
